# 02 — Model Training, Comparison & Ensemble
## NIDS — Network Intrusion Detection System

### Models trained:
**Tree Pipeline** (NO scaling — tree models don't need it):
1. Decision Tree
2. Random Forest
3. XGBoost
4. LightGBM
5. Voting Ensemble (RF + XGBoost)
6. Stacking Ensemble (RF + XGBoost → LR)

**Neural Network Pipeline** (StandardScaler REQUIRED):
7. MLPClassifier (Multi-Layer Perceptron)

### Key Design Rules:
- **Tree models** → raw features (no normalization needed, trees split on thresholds)
- **MLP / Neural Network** → StandardScaler applied (gradient-based, sensitive to scale)
- Missing values: already handled in clean_data() — no redundant imputation added
- Ranking: Macro F1 (handles class imbalance better than accuracy)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import time
import joblib
import os
warnings.filterwarnings('ignore')

from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import (RandomForestClassifier, VotingClassifier,
                                      StackingClassifier)
from sklearn.neural_network  import MLPClassifier          # ← Neural Network
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics         import (accuracy_score, f1_score, precision_score,
                                      recall_score, classification_report,
                                      confusion_matrix, ConfusionMatrixDisplay)
from imblearn.over_sampling  import SMOTE
from xgboost                 import XGBClassifier

try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False
    print('LightGBM not installed. Run: pip install lightgbm')

os.makedirs('../data/processed', exist_ok=True)
print('All libraries imported!')


LightGBM not installed. Run: pip install lightgbm
All libraries imported!


## Step 1 — Load & Clean Data


In [2]:
DATA_PATH = '../data/raw/cicids2017_cleaned.csv'
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

label_col = next((c for c in df.columns if 'label' in c.lower()), None)
print(f'Label column: "{label_col}"')

original_len = len(df)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# ── Check: are missing values already handled? ────────────────────────────
remaining_nan = df.isnull().sum().sum()
print(f'\nMissing values after clean: {remaining_nan}')
print('→ No redundant imputation needed (already handled above)')

print(f'Removed {original_len - len(df):,} bad rows. Remaining: {len(df):,}')
print(f'\nClass distribution:')
print(df[label_col].value_counts().to_string())


Loaded: 2,520,751 rows x 53 columns
Label column: "None"

Missing values after clean: 0
→ No redundant imputation needed (already handled above)
Removed 161 bad rows. Remaining: 2,520,590

Class distribution:


KeyError: None

## Step 2 — Preprocessing & Two Pipelines

```
Raw Data
    │
    ├── SMOTE (balance classes)
    │       │
    │       ├── Tree Pipeline (RF, XGBoost, LightGBM, DT)
    │       │       └── RAW features → NO scaling
    │       │
    │       └── Neural Network Pipeline (MLP)
    │               └── StandardScaler → scaled features
    │
    └── Test Set (raw) → scale with same scaler for NN eval
```


In [ ]:
X = df.drop(columns=[label_col])
y = df[label_col]

non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print(f'Dropping non-numeric columns: {non_numeric}')
    X.drop(columns=non_numeric, inplace=True)

# Label encoding
le = LabelEncoder()
y_encoded = le.fit_transform(y)
joblib.dump(le, '../label_encoder.pkl')
print(f'Label classes ({len(le.classes_)}): {list(le.classes_)}')

# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

# SMOTE — on training data only
print('\nApplying SMOTE...')
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
print(f'After SMOTE: {len(X_train_bal):,} training samples')

# Save test data
np.save('../data/processed/X_test.npy', X_test)
np.save('../data/processed/y_test.npy', y_test)
print('Test data saved.')


In [ ]:
# ═══════════════════════════════════════════════════════════
#  DUAL PIPELINE SETUP
# ═══════════════════════════════════════════════════════════

# Pipeline 1: Tree Models — RAW features (no normalization)
# Trees split on thresholds, so scale doesn't matter.
X_train_tree = X_train_bal          # raw balanced training data
X_test_tree  = X_test               # raw test data

# Pipeline 2: Neural Network — MUST scale
# Gradient descent is sensitive to feature scale.
nn_scaler = StandardScaler()
X_train_nn = nn_scaler.fit_transform(X_train_bal)   # fit on train only
X_test_nn  = nn_scaler.transform(X_test)             # apply to test

# Also save one scaler for the main inference pipeline (used by predict.py)
joblib.dump(nn_scaler, '../scaler.pkl')

print('Pipeline 1 (Tree): X_train_tree shape:', X_train_tree.shape)
print('Pipeline 2 (NN):   X_train_nn shape:  ', X_train_nn.shape)
print('NN features: mean ≈', round(X_train_nn.mean(), 4), '| std ≈', round(X_train_nn.std(), 4))
print('(Should be ≈ 0.0 and 1.0 respectively)')


## Step 3 — Helper Function & Results Store


In [ ]:
def train_and_evaluate(model, name, X_tr, y_tr, X_te, y_te, pipeline='tree'):
    """
    Train model, evaluate, and return results dict.
    pipeline: 'tree' or 'nn' — just for display info
    """
    pipeline_label = '🌳 Tree' if pipeline == 'tree' else '🧠 Neural'
    print(f'Training {pipeline_label} | {name}...')
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0

    y_pred = model.predict(X_te)
    acc    = accuracy_score(y_te, y_pred)
    f1_mac = f1_score(y_te, y_pred, average='macro',    zero_division=0)
    f1_wt  = f1_score(y_te, y_pred, average='weighted', zero_division=0)
    prec   = precision_score(y_te, y_pred, average='macro', zero_division=0)
    rec    = recall_score(y_te, y_pred, average='macro',    zero_division=0)

    print(f'  ✓ {train_time:.1f}s  |  Acc={acc*100:.2f}%  |  Macro F1={f1_mac*100:.2f}%')

    return {
        'Model':       name,
        'Pipeline':    pipeline_label,
        'Accuracy':    round(acc  * 100, 2),
        'Macro F1':    round(f1_mac * 100, 2),
        'Weighted F1': round(f1_wt  * 100, 2),
        'Precision':   round(prec * 100, 2),
        'Recall':      round(rec  * 100, 2),
        'Train Time':  f'{train_time:.1f}s',
        '_model':      model,
        '_y_pred':     y_pred,
        '_pipeline':   pipeline,
    }

results = []
print('Helper ready!')


## Step 4 — Train Tree Pipeline Models (NO scaling)


In [ ]:
# ── Decision Tree ─────────────────────────────────────────────────────────────
dt = DecisionTreeClassifier(max_depth=20, min_samples_split=5, random_state=42)
results.append(train_and_evaluate(
    dt, 'Decision Tree', X_train_tree, y_train_bal, X_test_tree, y_test, pipeline='tree'
))


In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=100, max_depth=20, n_jobs=-1, random_state=42, class_weight='balanced'
)
results.append(train_and_evaluate(
    rf, 'Random Forest', X_train_tree, y_train_bal, X_test_tree, y_test, pipeline='tree'
))


In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=42, eval_metric='mlogloss', verbosity=0
)
results.append(train_and_evaluate(
    xgb, 'XGBoost', X_train_tree, y_train_bal, X_test_tree, y_test, pipeline='tree'
))


In [ ]:
# ── LightGBM ──────────────────────────────────────────────────────────────────
if LGBM_AVAILABLE:
    lgbm_model = lgb.LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        n_jobs=-1, random_state=42, class_weight='balanced', verbose=-1
    )
    results.append(train_and_evaluate(
        lgbm_model, 'LightGBM', X_train_tree, y_train_bal, X_test_tree, y_test, pipeline='tree'
    ))
else:
    print('Skipping LightGBM (not installed). Run: pip install lightgbm')


## Step 5 — Neural Network Pipeline (MLPClassifier)

### Why separate pipeline?
- Tree models split on thresholds → scale doesn't affect splits → NO scaling needed
- Neural networks use gradient descent → large features dominate gradients → MUST scale

### Architecture:
```
Input (N features)
    → Dense(256, ReLU)
    → Dense(128, ReLU)
    → Dense(64, ReLU)
    → Dropout (via early stopping)
    → Softmax(num_classes)
```


In [ ]:
# ── MLPClassifier (Neural Network) ────────────────────────────────────────────
# Uses X_train_nn (StandardScaler applied) — separate from tree pipeline!

print('Training Neural Network (MLP) on SCALED features...')
print('Note: Tree models above used RAW features — correct by design.')

mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),  # 3-layer network
    activation='relu',                   # ReLU is best for tabular data
    solver='adam',                       # Adam optimizer
    learning_rate_init=0.001,
    max_iter=100,                        # epochs
    early_stopping=True,                 # stop when val loss stops improving
    validation_fraction=0.1,             # 10% of train for validation
    n_iter_no_change=10,                 # patience
    batch_size=512,
    random_state=42,
    verbose=False
)

# Uses NN pipeline (scaled data)
results.append(train_and_evaluate(
    mlp, 'Neural Network (MLP)', X_train_nn, y_train_bal, X_test_nn, y_test,
    pipeline='nn'
))

print(f'\nMLP converged in {mlp.n_iter_} iterations')
print(f'Best validation loss: {mlp.best_validation_score_:.4f}')


In [ ]:
# Plot MLP training loss curve
plt.figure(figsize=(10, 4))
plt.plot(mlp.loss_curve_, color='#e74c3c', label='Training Loss')
plt.xlabel('Iteration (Epoch)')
plt.ylabel('Loss')
plt.title('Neural Network (MLP) — Training Loss Curve', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../data/processed/mlp_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Loss curve saved.')


## Step 6 — Voting Ensemble (Tree Pipeline only)
Ensembles should only combine models from the SAME pipeline.
Mixing tree + NN in a VotingClassifier would require the same input preprocessing,
which adds complexity. Keep them separate for cleaner comparison.


In [ ]:
# Re-create fresh models for ensemble
rf_v  = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42)
xgb_v = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                       n_jobs=-1, random_state=42, eval_metric='mlogloss', verbosity=0)

voting = VotingClassifier(
    estimators=[('rf', rf_v), ('xgb', xgb_v)],
    voting='soft'   # soft voting uses probabilities → more accurate
)
# Uses tree pipeline (unscaled)
results.append(train_and_evaluate(
    voting, 'Voting (RF+XGB)', X_train_tree, y_train_bal, X_test_tree, y_test, pipeline='tree'
))


In [ ]:
# Stacking Ensemble
rf_s  = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42)
xgb_s = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                       n_jobs=-1, random_state=42, eval_metric='mlogloss', verbosity=0)
meta  = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)

stacking = StackingClassifier(
    estimators=[('rf', rf_s), ('xgb', xgb_s)],
    final_estimator=meta,
    cv=3, n_jobs=-1
)
results.append(train_and_evaluate(
    stacking, 'Stacking (RF+XGB→LR)', X_train_tree, y_train_bal, X_test_tree, y_test, pipeline='tree'
))


## Step 7 — Full Model Comparison Table (Ranked by Macro F1)


In [ ]:
cols = ['Model', 'Pipeline', 'Accuracy', 'Macro F1', 'Weighted F1', 'Precision', 'Recall', 'Train Time']
df_results = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in results])
df_results = df_results[cols].sort_values('Macro F1', ascending=False).reset_index(drop=True)
df_results.index += 1

print('=' * 90)
print('  MODEL COMPARISON — Ranked by Macro F1 (handles class imbalance best)')
print('=' * 90)
print(df_results.to_string())
print('=' * 90)
print(f'\n🏆 Best model: {df_results.iloc[0]["Model"]}  |  Macro F1: {df_results.iloc[0]["Macro F1"]}%')

nn_row = df_results[df_results['Model'].str.contains('Neural|MLP')]
if not nn_row.empty:
    print(f'\n🧠 Neural Network rank: #{nn_row.index[0]}  |  Macro F1: {nn_row.iloc[0]["Macro F1"]}%')


In [ ]:
# Grouped bar chart — Tree vs NN pipeline
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models     = df_results['Model'].tolist()
f1_scores  = df_results['Macro F1'].tolist()
accuracies = df_results['Accuracy'].tolist()
pipelines  = [r['_pipeline'] for r in sorted(results, key=lambda x: -x['Macro F1'])]

colors = ['#3498db' if p == 'tree' else '#e74c3c' for p in pipelines]

# F1 chart
bars = axes[0].barh(models[::-1], f1_scores[::-1], color=colors[::-1], alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Macro F1 (%)')
axes[0].set_title('Macro F1 by Model', fontsize=13, fontweight='bold')
axes[0].set_xlim(min(f1_scores)-5, 102)
for bar, val in zip(bars, f1_scores[::-1]):
    axes[0].text(val+0.3, bar.get_y()+bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)

# Accuracy chart
bars2 = axes[1].barh(models[::-1], accuracies[::-1], color=colors[::-1], alpha=0.85, edgecolor='white')
axes[1].set_xlabel('Accuracy (%)')
axes[1].set_title('Accuracy by Model', fontsize=13, fontweight='bold')
axes[1].set_xlim(min(accuracies)-5, 102)
for bar, val in zip(bars2, accuracies[::-1]):
    axes[1].text(val+0.3, bar.get_y()+bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(color='#3498db', label='🌳 Tree Pipeline (unscaled)'),
                   Patch(color='#e74c3c', label='🧠 Neural Network Pipeline (scaled)')]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=10)

plt.suptitle('All Models — Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('../data/processed/model_comparison_with_nn.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 8 — Confusion Matrix (Best Model)


In [ ]:
best_name   = df_results.iloc[0]['Model']
best_result = next(r for r in results if r['Model'] == best_name)
best_y_pred = best_result['_y_pred']

print(f'Best model: {best_name}')
cm = confusion_matrix(y_test, best_y_pred)
fig, ax = plt.subplots(figsize=(14, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, xticks_rotation=45, colorbar=True, cmap='Blues')
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/confusion_matrix_best.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nClassification Report — {best_name}')
print(classification_report(y_test, best_y_pred, target_names=le.classes_, zero_division=0))


## Step 9 — Save Best Model


In [ ]:
best_model    = best_result['_model']
best_pipeline = best_result['_pipeline']

joblib.dump(best_model, '../model.pkl')

print('=' * 60)
print('  TRAINING COMPLETE')
print('=' * 60)
print(f'  Best model     : {best_name}')
print(f'  Pipeline       : {best_pipeline} ({"No scaling" if best_pipeline == "tree" else "StandardScaler applied"})')
print(f'  Accuracy       : {df_results.iloc[0]["Accuracy"]}%')
print(f'  Macro F1       : {df_results.iloc[0]["Macro F1"]}%')
print()
print('  Saved:')
print('    ../model.pkl           — best model')
print('    ../scaler.pkl          — StandardScaler (for NN pipeline & inference)')
print('    ../label_encoder.pkl   — LabelEncoder')
print('    ../data/processed/     — charts, confusion matrix, loss curve')
print('=' * 60)

print('\nAll model results:')
print(df_results[['Model', 'Pipeline', 'Accuracy', 'Macro F1']].to_string())
